<a href="https://colab.research.google.com/github/chuanbinp/uncertainty-aware-inference/blob/master/TeamB/teamb_profiler_only.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Team B — PyTorch Profiler
## Mistral-7B PTQ Sweep

Runs **PyTorch Profiler + Roofline analysis** across all five quantization configs.

Each config produces:
- `{config}_profile.json` — timing, memory, kernel breakdown, roofline numbers
- `{config}_chrome.json` — Chrome trace (open at [perfetto.dev](https://ui.perfetto.dev))

**AWQ note:** AWQ custom CUDA kernels bypass the PyTorch dispatcher and will not
appear in the kernel table — this is expected. Timing and memory are still accurate.

**To adapt for your model:** update `ALL_CONFIGS` and `REPO_DIR` in Section 3,
add your model entries to `MODEL_REGISTRY` in `run_profiler.py`, and add your AWQ
config key to `CUSTOM_KERNEL_CONFIGS` in that same file.


<a href="https://colab.research.google.com/github/chuanbinp/uncertainty-aware-inference/blob/master/TeamB/teamb_vllm_profiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Setup

### 1.1 Verify GPU

In [ ]:
import subprocess, sys
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM            : {total_mem:.1f} GB")
    if total_mem < 20:
        print("WARNING: <20 GB VRAM — some configs may OOM. Recommend A100.")


### 1.2 Clone repo

In [ ]:
!git clone https://github.com/chuanbinp/uncertainty-aware-inference.git

### 1.3 Install core dependencies

In [ ]:
# Same package set as mistral_7b.ipynb — versions not changed
!pip install numpy plotly matplotlib torch transformers datasets accelerate gptqmodel netcal autoawq bitsandbytes


### 1.4 Install vLLM

vLLM is installed separately after core deps. `vllm==0.4.3` is compatible with the torch version above.

In [ ]:
!pip install vllm==0.4.3

### 1.5 Install W&B

In [ ]:
!pip install wandb

### 1.6 Verify scripts are present

In [ ]:
import os
REPO_DIR = "/content/uncertainty-aware-inference/TeamB"
for script in ["run_vllm.py", "run_profiler.py", "run_eval.py", "configs.py"]:
    path = os.path.join(REPO_DIR, script)
    status = "OK" if os.path.exists(path) else "MISSING — upload manually"
    print(f"  {script}: {status}")


# 2. Authentication

### 2.1 HuggingFace — required for gated Mistral-7B weights

In [ ]:
from huggingface_hub import login
from getpass import getpass
import os

HF_TOKEN = getpass("Enter your HuggingFace token: ")
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN   # propagate to subprocesses
print("HuggingFace login successful")


### 2.2 Weights & Biases

In [ ]:
import wandb
from getpass import getpass

WANDB_API_KEY = getpass("Enter your W&B API key (Enter to skip): ")
if WANDB_API_KEY.strip():
    wandb.login(key=WANDB_API_KEY)
    WANDB_ENABLED = True
    print("W&B login successful")
else:
    WANDB_ENABLED = False
    print("W&B disabled — results will be saved to JSON only")


# 3. Experiment Configuration

In [ ]:
import os
from pathlib import Path

REPO_DIR     = "/content/uncertainty-aware-inference/TeamB"
PROFILER_OUT = "/content/profiler_results"

for d in [PROFILER_OUT]:
    os.makedirs(d, exist_ok=True)

# All configs to sweep — same ordering as configs.py MODEL_REGISTRY
ALL_CONFIGS = [
    "mistral-7b-fp16",
    "mistral-7b-gptq-int4",
    "mistral-7b-gptq-int8",
    "mistral-7b-awq-int4",
    "mistral-7b-nf4",
]

WANDB_PROJECT = "UAI_Project"
WANDB_ENTITY  = "Uncertainty_Aware_Inference_Lab"

print(f"Sweep configs : {ALL_CONFIGS}")
print(f"Profiler out  : {PROFILER_OUT}")
print(f"W&B enabled   : {WANDB_ENABLED}")


# 4. Run Helpers

In [ ]:
import subprocess, sys, json, os
from pathlib import Path

def run_script(script_path: str, extra_args: list):
    """Run a Python script as a subprocess, streaming stdout + stderr."""
    cmd = [sys.executable, script_path] + extra_args
    print(f"\nCMD: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        # Print last 3000 chars of stderr (avoids flooding for large models)
        stderr_tail = result.stderr[-3000:]
        print("STDERR (tail):", stderr_tail)
    return result.returncode == 0

def load_json(path: str) -> dict | None:
    p = Path(path)
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)

def wandb_run_init(config_key: str, experiment: str):
    """Create and return a W&B run if enabled, else None."""
    if not WANDB_ENABLED:
        return None
    return wandb.init(
        project  = WANDB_PROJECT,
        entity   = WANDB_ENTITY,
        name     = f"teamB_{config_key}_{experiment}",
        reinit   = True,
        config   = {"model": "mistral-7b", "team": "team-b",
                    "config_key": config_key, "experiment": experiment},
    )

print("Helpers ready.")


# 6. PyTorch Profiler

Calls `run_profiler.py` for each config. The script:
- Loads the model using `gptqmodel` (GPTQ), `autoawq` with `fuse_layers=False` (AWQ),
  `bitsandbytes` (NF4), or vanilla HF (FP16)
- Runs 3 warmup + 5 profiled passes with 50 output tokens
- Captures: CUDA kernel breakdown, memory stats, FLOPs, Chrome trace
- Saves `{config_key}_profile.json` and `{config_key}_chrome.json` to `PROFILER_OUT`

> The profiler models are loaded via the same package versions as the calibration notebook —
> **no version changes**.


In [ ]:
import json
from pathlib import Path

SCRIPT_PROF = os.path.join(REPO_DIR, "run_profiler.py")
prof_summary = {}

for config_key in ALL_CONFIGS:
    print(f"\n{'#'*65}")
    print(f"# Profiler: {config_key}")
    print(f"{'#'*65}")

    run = wandb_run_init(config_key, "pytorch_profiler")
    run_id = run.id if run else None

    args = [
        "--config",     config_key,
        "--output-dir", PROFILER_OUT,
        "--hf-token",   HF_TOKEN,
    ]
    if run_id:
        args += ["--wandb-run-id", run_id, "--wandb-project", WANDB_PROJECT]

    ok = run_script(SCRIPT_PROF, args)

    result = load_json(f"{PROFILER_OUT}/{config_key}_profile.json")
    if result:
        prof_summary[config_key] = result
        t  = result["timing"]
        m  = result["memory"]
        rl = result["roofline"]
        print(f"DONE {config_key}: {t['tok_per_sec']:.1f} tok/s | "
              f"{m['peak_gpu_gb']:.2f} GB | AI(decode)={rl['ai_decode']:.1f} [{rl['bound_decode']}]")
        if run:
            run.log({
                "profiler/tok_per_sec":      t["tok_per_sec"],
                "profiler/avg_inference_ms": t["avg_inference_ms"],
                "profiler/peak_gpu_gb":      m["peak_gpu_gb"],
                "profiler/param_gb":         m["param_gb"],
                "profiler/total_flops":      result["compute"]["total_flops"],
                "roofline/ai_decode":        rl["ai_decode"],
                "roofline/ai_prefill":       rl["ai_prefill"],
                "roofline/bound_decode":     rl["bound_decode"],
            })
            if result.get("top_kernels"):
                kernel_table = wandb.Table(
                    columns=["name", "cuda_time_ms", "pct", "calls"],
                    data=[[k["name"], k["cuda_time_ms"], k["pct"], k["calls"]]
                          for k in result["top_kernels"]],
                )
                run.log({"profiler/top_kernels": kernel_table})
    else:
        print(f"WARN: no JSON for {config_key} — see stderr above")

    if run:
        run.finish()

print("\nProfiler complete for all configs.")


### 6.1 Profiler Results Summary

In [ ]:
COLS = ["Config", "Tok/s", "Peak GB", "Param GB", "AI decode", "Bound"]
rows = []
for cfg in ALL_CONFIGS:
    d = load_json(f"{PROFILER_OUT}/{cfg}_profile.json")
    if d:
        t  = d["timing"]
        m  = d["memory"]
        rl = d["roofline"]
        rows.append([cfg, f"{t['tok_per_sec']:.1f}", f"{m['peak_gpu_gb']:.3f}",
                     f"{m['param_gb']:.3f}", f"{rl['ai_decode']:.2f}", rl["bound_decode"]])
    else:
        rows.append([cfg, "N/A", "N/A", "N/A", "N/A", "N/A"])

w = [30, 9, 9, 10, 11, 8]
header = "  ".join(f"{c:<{w[i]}}" for i, c in enumerate(COLS))
print(header)
print("-" * (sum(w) + len(w)*2))
for r in rows:
    print("  ".join(f"{v:<{w[i]}}" for i, v in enumerate(r)))


### 6.2 Top 5 CUDA Kernels per Config

In [ ]:
for cfg in ALL_CONFIGS:
    d = load_json(f"{PROFILER_OUT}/{cfg}_profile.json")
    if not d:
        continue
    print(f"\n  {cfg}")
    print(f"  {'Kernel':<55} {'CUDA ms':>10}  {'%':>7}  {'calls':>8}")
    print(f"  {'-'*85}")
    for k in d.get("top_kernels", [])[:5]:
        print(f"  {k['name'][:55]:<55} {k['cuda_time_ms']:>10.3f}  {k['pct']:>7.1f}  {k['calls']:>8}")


# 8. Roofline Plot

In [ ]:
import json, math
import matplotlib.pyplot as plt
from pathlib import Path

fig, ax = plt.subplots(figsize=(10, 6))

A100_TFLOPS_FP16  = 312.0   # TFLOPS FP16
A100_BANDWIDTH_TB = 1.555   # TB/s HBM2 memory bandwidth
 
# Keep legacy names so nothing else breaks
L4_TFLOPS  = A100_TFLOPS_FP16
L4_BW = A100_BANDWIDTH_TB

ridge       = L4_TFLOPS / L4_BW   # FLOPs/Byte

ai_pts = [0.1, ridge, 10_000]
roof_y = [min(a * L4_BW, L4_TFLOPS) / 1e12 for a in ai_pts]
ax.plot(ai_pts, roof_y, "k-", linewidth=2, label="L4 Roofline (FP16)")
ax.axvline(ridge, color="gray", linestyle="--", alpha=0.5, label=f"Ridge point ≈{ridge:.0f}")

COLORS = {
    "mistral-7b-fp16":       "#2196F3",
    "mistral-7b-gptq-int4":  "#4CAF50",
    "mistral-7b-gptq-int8":  "#FF9800",
    "mistral-7b-awq-int4":   "#9C27B0",
    "mistral-7b-nf4":        "#F44336",
}
LABELS = {
    "mistral-7b-fp16":       "FP16",
    "mistral-7b-gptq-int4":  "GPTQ INT4",
    "mistral-7b-gptq-int8":  "GPTQ INT8",
    "mistral-7b-awq-int4":   "AWQ INT4",
    "mistral-7b-nf4":        "NF4",
}

for cfg in ALL_CONFIGS:
    d = load_json(f"{PROFILER_OUT}/{cfg}_profile.json")
    if not d:
        continue
    rl  = d["roofline"]
    tps = d["timing"]["tok_per_sec"]
    # Approximate TFLOPS from tok/s and param count
    param_gb = d["memory"]["param_gb"]
    bits_map = {"fp16": 16, "gptq-int8": 8, "gptq-int4": 4, "awq-int4": 4, "nf4": 4}
    bits = bits_map.get(cfg.replace("mistral-7b-", ""), 16)
    n_params  = param_gb * 1e9 / (bits / 8)
    achieved_tflops = tps * 2 * n_params / 1e12

    col = COLORS[cfg]
    # Decode point (circle)
    ax.scatter([rl["ai_decode"]], [achieved_tflops], color=col, s=130,
               zorder=5, label=f"{LABELS[cfg]}")
    # Prefill point (triangle)
    ax.scatter([rl["ai_prefill"]], [achieved_tflops], color=col, s=80,
               marker="^", zorder=5)
    ax.annotate(LABELS[cfg], xy=(rl["ai_decode"], achieved_tflops),
                xytext=(8, 4), textcoords="offset points",
                color=col, fontsize=8.5, fontweight="bold")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Arithmetic Intensity (FLOPs/Byte)", fontsize=12)
ax.set_ylabel("Performance (TFLOPS)", fontsize=12)
ax.set_title("Roofline Model — Mistral-7B PTQ Sweep on L4\n"
             "circle = decode phase, triangle = prefill phase", fontsize=13)
ax.legend(fontsize=8.5, loc="upper left")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()

plot_path = "/content/roofline_mistral7b.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {plot_path}")

if WANDB_ENABLED:
    with wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                    name="teamB_roofline", reinit=True) as _run:
        _run.log({"roofline_plot": wandb.Image(plot_path)})
    print("Roofline plot logged to W&B")


# 7. Log Summary to W&B

In [ ]:
if WANDB_ENABLED:
    import wandb, json
    from pathlib import Path

    with wandb.init(
        project=WANDB_PROJECT, entity=WANDB_ENTITY,
        name="teamB_mistral7b_profiler_sweep",
        reinit="finish_previous",
        config={"model": "mistral-7b", "team": "team-b",
                "experiment": "profiler_sweep", "configs": ALL_CONFIGS},
    ) as _run:
        cols = ["config", "prof_tok_per_sec", "prof_peak_gpu_gb",
                "param_gb", "roofline_ai_decode", "roofline_bound"]
        rows = []
        for cfg in ALL_CONFIGS:
            pd = load_json(f"{PROFILER_OUT}/{cfg}_profile.json")
            row = [
                cfg,
                pd["timing"]["tok_per_sec"]    if pd else "N/A",
                pd["memory"]["peak_gpu_gb"]    if pd else "N/A",
                pd["memory"]["param_gb"]       if pd else "N/A",
                pd["roofline"]["ai_decode"]    if pd else "N/A",
                pd["roofline"]["bound_decode"] if pd else "N/A",
            ]
            rows.append(row)
        table = wandb.Table(columns=cols, data=rows)
        _run.log({"mistral7b_profiler_sweep": table})
        if os.path.exists("/content/roofline_mistral7b.png"):
            _run.log({"roofline_plot": wandb.Image("/content/roofline_mistral7b.png")})
        print("Summary logged to W&B.")
else:
    print("W&B disabled.")


# 8. Download Results

In [ ]:
!zip -r /content/profiler_results.zip /content/profiler_results/
print("Zipped profiler_results.zip")


In [ ]:
from google.colab import files
import os
files.download("/content/profiler_results.zip")
if os.path.exists("/content/roofline_mistral7b.png"):
    files.download("/content/roofline_mistral7b.png")
